# Imports

In [8]:
# ----------------- Load packages ------------------- #
import os
import contextlib
import io
import numpy as np
import pandas as pd
import mne
from scipy.io import loadmat
import warnings
import matplotlib.pyplot as plt
import time
from pathlib import Path
from datetime import datetime




from somnopy import *
from somnopy.somno import get_sosp, get_sosp_for_folder
from somnopy.event_detection import SO_detection, SP_detection, detect_swa
from somnopy.metrics import pac,event_lock
from somnopy.utils import set_up_raw
from somnopy.polysomnography import PolySomnoGraphy
import yasa
from lspopt import spectrogram_lspopt



# Params

In [2]:
###The following are REQUIRED PARAMETERS:

# data_path should point to a folder of folders. Each folder within data_path will have its .edf and .txt / .mat contents analyzed separately
# output_path should point to a folder where you want to store your results. 

# Subfolders within output_path will match the names of the subfolders within data_path
# If matching subfolders have not been created yet, they will be created automatically 
# Each subfolder will store results for the corresponding .edf files in data_path

# Group data into subfolders however you would like. If you want metrics stored per participant, make sure each .edf and .txt / .mat
# file is stored in it's own subfolder in data_path per participant. If you want metrics stored across all participants, just
# create one subfolder within data_path for all .edf and .txt files

# Make sure each .edf filename matches its corresponding .txt / .mat file

data_path = r"C:\Users\Beast\Ryan\somnopy\somnopy\data"
output_path = r"C:\Users\Beast\Ryan\somnopy\somnopy\output"

#get the folder paths to pass into the get_sosp_for_folder function
folder_paths = [str(p) for p in Path(data_path).iterdir() if p.is_dir()]


###The following are OPTIONAL PARAMETERS, the default value is what you see, update as necessary:
interest_stage = ('N2', 'SWS') #sleep stages interested in evaluating
sp_method = 'Hahn2020' #Method used for spindle detection. See available methods below
so_method = 'Staresina' #Method used for slow oscillation detection. See available methods below
coupling = True #If set to true, additional metrics such as PAC, or spindle to slow oscillation coupling percentage will be calulated and plotted
swa = True #include swa in final csv file
scoring_dur = 30 # (in seconds) the duration of each epoch used for sleep scoring
rereference = False # If reference channels are not contralateral mostoids or you are unsure, otherwise set to: rereference = ['M1', 'M2'], also optional to set 'average' to use the average of all channels as reference
chan_limit = ['F3', 'C3', 'F4', 'C4'] # Use None to process all channels, or use ['Fz', 'Cz', 'F3', 'C3', ...] to process selected channels only.
channels_to_drop = ('E1', 'E2', 'ChinC', 'ChinL', 'ChinR') #by default, will drop non-eeg channels. Add any channels that you would want to drop completely
montage_temp = "standard_1005" # Use standard_1005 for mid and high-density caps, standard_1020 for low-density caps, or specify another montage. See other available montages below
baseline = True # Baseline correction before the event detection
verbose = True # Additional diagnostic plotting/ printing if set to true
#The following are customisable Slow Oscillatoin detection metrics, that have different default depending on method used
filter_freq = None
duration = None
filter_type = 'fir'
#The following are customisable Spindle detection metrics, that have different default depending on method used
l_freq = None
h_freq = None
dur_lower = None
dur_upper = None


In [4]:
folder_paths = folder_paths[1:]

In [5]:
folder_paths

['C:\\Users\\Beast\\Ryan\\somnopy\\somnopy\\data\\participant1',
 'C:\\Users\\Beast\\Ryan\\somnopy\\somnopy\\data\\participant2',
 'C:\\Users\\Beast\\Ryan\\somnopy\\somnopy\\data\\preproc',
 'C:\\Users\\Beast\\Ryan\\somnopy\\somnopy\\data\\scoring']

# Coupling Analysis

In [ ]:
#run coupling analysis
#figs should auto save out
#figs now save, future should stem all info (datetime included) and pass to function calls. Currently overwrites past figs

today_str = datetime.now().strftime("%Y-%m-%d")


#run through all participant folders
for folder in folder_paths:
    folder_path = Path(folder)
    folder_name = folder_path.name
    output_folder = Path(output_path) / folder_name 
    output_folder.mkdir(parents=True, exist_ok=True)

    print(f"RUNNING {folder}")
    
    #try running SOSP
    try:
        start_time = time.time()
        #run analysis function
        event_summary_all, coupling_event_all, so_waveform_all = get_sosp_for_folder(
            folder,
            folder,
            chan_limit=chan_limit,
            coupling=coupling,
            swa=swa,
            interest_stage=interest_stage,
            sp_method=sp_method,
            so_method=so_method,
            scoring_dur=scoring_dur,
            outpath = output_folder
        )
        
        #save summary for each eeg file within single participant as csv.
        for filename, summary in event_summary_all.items():
            stem = Path(filename).stem  
            csv_path = output_folder / f"{stem}_somnopy_summary_{today_str}.csv"

            summary.to_csv(csv_path, index=False)
            print(f"📄 Saved: {csv_path}")
            
        end_time = time.time()
        duration = end_time - start_time
        print(f"✅ Finished {folder} in {int(duration//60)}m {int(duration%60)}s")

    except Exception as e:
        print(f"❌ Error processing {folder}: {e}")

    #uncomment if you want to try just the first folder
    break 


    


# YASA: Spectrogram and Sleep Staging on EDF files

## Functions for plotting

In [ ]:
#load in yasa data from edf and mat files TXT AND MAT PROCESSING IMPLEMENTED
def load_yasa_from_edf(edf_path, hyp_path, stage_mapping):

    #load raw data
    raw = mne.io.read_raw_edf(edf_path, preload=True)
    raw_data, times = raw.get_data(return_times=True)
    channels = raw.ch_names[:len(raw_data)] #channels are validated to be in proper order
    sampling_rate = raw.info["sfreq"]


    #load hypnogram
    if hyp_path.endswith(".mat"):
        mat = loadmat(hyp_path)

        #store hypnogram data, window size, and sampling rate
        hyp_data = mat["stageData"]["stages"][0, 0].squeeze().astype(int)
        hyp_window = int(mat["stageData"]['win'][0][0][0][0])
        hyp_srate = int(mat["stageData"]['srate'][0][0][0][0])

        #MATLAB TO YASA STAGE CODES
        stage_mapping = {
            0: 0,
            1: 1,
            2: 2,
            3: 3,
            5: 4,
            6: -1,
            7: -2
        }

        #convert the integer codes present in hyp_data into YASA approved integer codes
        hyp_yasa = np.array([stage_mapping[x] for x in hyp_data], dtype=int)

    elif hyp_path.endswith(".txt"):
        
        #start parsing the txt hypnogram line by line
        with open(hyp_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()

        header_idx = None
        for i, line in enumerate(lines):
            if line.startswith("Sleep Stage"):

                header_idx = i
                break

        if header_idx is None:
            raise ValueError("Could not find table header in file")

        #store hypnogram as a pandas object
        df = pd.read_csv(
            hyp_path,
            sep="\t",
            skiprows=header_idx,
            engine="python"
        )

        #remove spaces from column names
        df.columns = df.columns.str.strip()

        #keeping ONLY stages that are listed in the key, otherwise yasa cant parse it.
        df["Event"] = df["Event"].astype(str).str.strip().str.upper()
        df_stage = df[df["Event"].isin(stage_mapping.keys())].copy()
        if len(df_stage) == 0:
            raise ValueError("No valid sleep stage rows found")

        #convert to a numpy arr (does this do that?)
        hyp_yasa = df_stage["Event"].map(stage_mapping).values

        #store hypnogram window and sampling rate
        hyp_window = df_stage["Duration[s]"].astype(float)[0]
        hyp_srate = 1.0 / hyp_window

    #make data accessible by channel name. To access data for a certain channel, use input[channel]
    input = {}
    for channel, data in zip(channels, raw_data):
        input[channel] = data

    #upsample hypnogram data to EEG data (make each the same length)
    hypno = yasa.hypno_upsample_to_data(
        hypno=hyp_yasa,
        sf_hypno=1/30,
        data=raw_data[0], #potentially different lengths for different electrodes, NOTE in case of error
        sf_data=sampling_rate
    )


    return input, channels, hypno, sampling_rate, hyp_window

#plot a single electrode for a given yasa object
def plot_single_electrode(electrode, input, hypno, sf, hyp_window, verbose=False, outpath=None, filename="participant", fmin=0.5, fmax=25):


    fig = yasa.plot_spectrogram(input[electrode], hypno=hypno, sf=sf, win_sec=hyp_window, fmin=fmin, fmax=fmax) #potentially need eeg sampling rate not hypno sampling rate? should usually be the same

    if verbose:
        plt.show()

    if outpath is not None:
        fig.savefig(f"{outpath}\\{filename}_{electrode}_spectrogram.png")


    return fig

#plots all electrodes for given yasa object
def plot_all_electrodes(input, hypno, sf, hyp_window, verbose=False, outpath=None, filename="participant", fmin=0.5, fmax=25):
    for electrode in input:
        plot_single_electrode(electrode, input, hypno, sf, hyp_window, verbose=verbose, outpath=outpath, filename="participant", fmin=fmin, fmax=fmax)
        print('complete')

#calculate spectrogram data
#NOTE uses the same spectrogram calculation methods as yasa, but this isn't technically the data in the outputted spectrogram, should be identical
#yasa does not output data with their spectrogram so I had to re implement it's logic
def save_spectrogram_csv(electrode, input, sampling_rate, win_sec=30, outpath=None, filename="participant", fmin=0.5, fmax=25):

    nperseg = int(win_sec * sampling_rate)
    frequencies, times, powers = spectrogram_lspopt(input[electrode], sampling_rate, nperseg=nperseg, noverlap=0)

    #convert to dB / Hz
    power_db = 10 * np.log10(powers)

    #keep requested frequency range
    keep = (frequencies >= fmin) & (frequencies <= fmax)
    frequencies = frequencies[keep]
    power_db = power_db[keep, :]

    print(f"frequency shape: {frequencies.shape}")
    print(f"times shape: {times.shape}")
    print(f"power decibel shape: {power_db.shape}")

    df = pd.DataFrame(
        power_db,
        index=frequencies,
        columns=times
    )

    df.index.name = "frequency_hz"
    df.columns.name = "time_sec"

    if outpath is not None:
        df.to_csv(f"{outpath}\\{filename}_{electrode}_power_data.csv")

    return df

def automatic_sleep_staging(edf_path, eeg="C4", eog=None, emg=None, outpath=None, filename="participant"):
    # Load an EDF file using MNE
    raw = mne.io.read_raw_edf(edf_path, preload=True)
    # Initialize the sleep staging instance
    sls = yasa.SleepStaging(
        raw,
        eeg_name=eeg,
        eog_name=eog,
        emg_name=emg

    )
    # Print some basic info
    #sls
    # Get the predicted sleep stages
    hyp = sls.predict()

    #hyp.hypno
    #Get the predicted probabilities
    probs = hyp.proba
    # Get the confidence
    confidence = hyp.proba.max(axis=1)


    df = hyp.hypno.to_frame(name="stage").reset_index()
    df["time_sec"] = (df["Time"] - df["Time"].iloc[0]).dt.total_seconds()

    if outpath is not None:
        df.to_csv(f"{outpath}\\{filename}_sleep_stages.csv", index=False)

        # Plot the predicted probabilities
        ax = sls.plot_predict_proba()
        fig = ax.get_figure()
        fig.savefig(f"{outpath}\\{filename}_sleep_stage_probabilities.png")


    return df, hyp, probs, confidence

def run_sleep_staging_on_folder(input_folder, out_folder):
    input_folder = Path(input_folder)
    out_folder = Path(out_folder)

    out_folder.mkdir(parents=True, exist_ok=True)

    failed_files = []

    # iterate through all EDF files
    edf_files = list(input_folder.glob("*.edf"))

    print(f"Found {len(edf_files)} EDF files.\n")

    for edf_file in edf_files:
        print(f"Processing: {edf_file.name}")

        try:
            # create per-file output directory (optional but recommended)
            file_out = out_folder / edf_file.stem
            file_out.mkdir(exist_ok=True)

            df, hyp, probs, confidence = automatic_sleep_staging(
                str(edf_file),
                outpath=str(file_out)
            )

            print(f"✅ Success: {edf_file.name}\n")

        except Exception as e:
            print(f"❌ Failed: {edf_file.name}")
            print(f"   Error: {e}\n")

            failed_files.append((edf_file.name, str(e)))
            continue

    # summary
    print("\n===== SUMMARY =====")
    print(f"Total files: {len(edf_files)}")
    print(f"Failed: {len(failed_files)}")

    if failed_files:
        print("\nFailed files:")
        for name, err in failed_files:
            print(f"- {name}: {err}")

    return failed_files

## Load in edf and hypnogram data

In [ ]:
#change edf and hypnogram paths you would like to analyze

#edf_path = r"Y:\SNL\EmoCuing\EC\PSG\input\all_files\EmoCuing_0101.edf" 
#hypnogram_path = r"Y:\SNL\EmoCuing\EC\PSG\input\all_files\EmoCuing_0101.mat" 
edf_path = r"Y:\SNL\HSR\PSG data\2Preproccessed\HSR103_W1_V1_Overnight\HSR103_W1_V1_Overnight.edf"
hypnogram_path = r"Y:\SNL\HSR\PSG data\2Preproccessed\HSR103_W1_V1_Overnight\HSR103_W1_V1_Overnight.txt"

#yasa uses specific codes that map to specific stages of sleep. 
#verify which codes in our matlab hypnograms and txt hypnograms correspond to which yasa sleep stages. 
#the strings on the lefthand side are what is present in the txt and mat files
#the numbers on the right correspond to yasa codes
#change the strings on the left to match what is in the mat and txt files
#if they are numbers, remove the quotes. If they are letters, keep the quotes. EX: "W" vs 1

#NOTE if we would like to include stages that arent present in the yasa codes, I can rewrite the spetrogram code to not utilize yasa at all

stage_mapping = {
    "W": 0,
    "N1": 1,
    "N2": 2,
    "N3": 3,
    "R": 4,
    "Spindle": -2,
}


input, channels, hypno, sampling_rate, hyp_window = load_yasa_from_edf(edf_path, hypnogram_path, stage_mapping)

Extracting EDF parameters from Y:\SNL\HSR\PSG data\2Preproccessed\HSR103_W1_V1_Overnight\HSR103_W1_V1_Overnight.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 10210999  =      0.000 ... 20421.998 secs...


06-Apr-26 12:40:10 | WARNING | Hypnogram is LONGER than data by 38.00 seconds. Cropping hypnogram to match data.size.


In [ ]:
#examine channel names
channels

## Plot Spectrogram for a single electrode

In [ ]:
electrode = 'A1' #specify which electrode you want to examine
verbose = True #do you want to see the plot in the notebook or just save it to outpath? Set to True if you want to see it
outpath = r"C:\Users\Beast\Ryan\somnopy\somnopy\somnopy_v2\output" #specify the folder to save to 
filename = Path(edf_path).stem #what do you want to call your output file? 
fmin = 0.5 #OPTIONAL: what's the lowest frequency you would like to include in your spectrogram?
fmax=4 #OPTIONAL: what's the highest frequency you would like to include in your spectrogram?

plot_single_electrode(electrode, input, hypno, sampling_rate, hyp_window, verbose=verbose, outpath=outpath, filename=filename, fmin=fmin, fmax=fmax)

## Plot Spectrogram for all electrodes

In [ ]:
verbose = True #do you want to see the plot in the notebook or just save it to outpath? Set to True if you want to see it
outpath = r"C:\Users\Beast\Ryan\somnopy\somnopy\somnopy_v2\output" #specify the folder to save to 
filename = Path(edf_path).stem #what do you want to call your output file? 
fmin = 0.5 #OPTIONAL: what's the lowest frequency you would like to include in your spectrogram?
fmax=4 #OPTIONAL: what's the highest frequency you would like to include in your spectrogram?

plot_all_electrodes(input, hypno, sampling_rate, hyp_window, verbose=True, outpath=outpath, filename=filename, fmin=fmin, fmax=fmax)

## Get spectrogram data as a CSV

In [ ]:
electrode = 'A1' #specify which electrode you want to examine
outpath = r"C:\Users\Beast\Ryan\somnopy\somnopy\somnopy_v2\output" #specify the folder to save to 
filename = Path(edf_path).stem #what do you want to call your output file? 
fmin = 0.5 #OPTIONAL: what's the lowest frequency you would like to include in your spectrogram?
fmax=4 #OPTIONAL: what's the highest frequency you would like to include in your spectrogram?

#uses the same spectrogram calculation methods as yasa
save_spectrogram_csv(electrode, input, sampling_rate, win_sec=hyp_window, outpath=outpath, filename=filename, fmin=fmin, fmax=fmax)


frequency shape: (106,)
times shape: (680,)
power decibel shape: (106, 680)


time_sec,15.0,45.0,75.0,105.0,135.0,165.0,195.0,225.0,255.0,285.0,...,20115.0,20145.0,20175.0,20205.0,20235.0,20265.0,20295.0,20325.0,20355.0,20385.0
frequency_hz,,,,,,,,,,,,,,,,,,,,,
0.500000,-99.046639,-89.669729,-91.540642,-91.677306,-95.380120,-97.058185,-94.990617,-98.097605,-87.297303,-103.219868,...,-96.566929,-98.450941,-97.911323,-97.334901,-98.298604,-97.360207,-96.767042,-96.135002,-99.497355,-91.021147
0.533333,-99.113411,-91.604468,-91.666603,-92.853781,-95.367007,-98.164118,-96.148125,-98.380470,-88.016123,-104.091296,...,-96.320091,-98.136854,-98.421111,-96.136201,-97.372487,-97.548867,-95.973560,-96.184949,-100.086437,-91.918121
0.566667,-98.431118,-93.690304,-91.668499,-93.669692,-95.250315,-98.534856,-97.779633,-98.606091,-88.742759,-104.407774,...,-96.216220,-98.070262,-99.110917,-95.567572,-96.787983,-97.070831,-95.609222,-96.527800,-99.374982,-92.280026
0.600000,-97.970446,-95.068527,-92.032612,-94.011883,-95.280289,-99.393782,-99.838140,-98.725446,-89.511862,-104.764797,...,-96.245845,-97.903104,-100.185996,-95.519331,-96.787819,-96.587795,-95.592840,-97.256188,-98.389640,-92.928267
0.633333,-97.532092,-95.565769,-92.373467,-93.661419,-95.477915,-99.178761,-102.065895,-98.993977,-90.336846,-104.740336,...,-96.203025,-97.784699,-101.753169,-96.057364,-97.406365,-96.227056,-95.997906,-97.798845,-97.311000,-93.729258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3.866667,-107.517201,-105.450306,-107.434084,-110.695495,-111.076092,-111.862836,-106.575257,-116.172673,-109.771311,-111.865430,...,-97.691812,-106.732673,-103.669458,-106.863255,-105.739915,-107.312390,-107.736428,-104.614940,-108.819737,-104.024979
3.900000,-108.089859,-105.906485,-107.107806,-110.457695,-111.602541,-111.679877,-107.659577,-115.501122,-110.321439,-110.681055,...,-97.421769,-108.000565,-103.287924,-107.598723,-104.987014,-108.242308,-109.942070,-104.557113,-108.258456,-104.348292
3.933333,-109.059699,-106.535113,-107.003797,-110.405099,-111.546923,-110.979865,-108.921059,-114.332534,-110.856101,-109.639153,...,-97.617966,-108.448627,-103.542562,-107.678003,-104.506110,-109.393303,-110.476402,-104.836628,-107.924097,-104.420118


## Automatic Sleep Staging for one EDF (still being worked on)

In [ ]:
test_path = r"Y:\SNL\MBHD\PSG\edf\MBHD08.edf" #change path to edf file you want analyzed
outpath = r"C:\Users\Beast\Ryan\somnopy\somnopy\somnopy_v2\output" #change path to output folder to store data

df, hyp, probs, confidence = automatic_sleep_staging(test_path, outpath=outpath)

## Automatic Sleep Staging for a folder of EDFs

In [ ]:
input_path = r"Y:\SNL\MBHD\PSG\edf" #change path to folder of edf files you would like analyzed
outpath = r"C:\Users\Beast\Ryan\somnopy\somnopy\somnopy_v2\output\MBHD_scoring" #change path to output folder to store data

run_sleep_staging_on_folder(input_path, outpath)